# 03 – Évaluation critique et interprétation du modèle

## Objectif du notebook

Ce notebook évalue et interprète le modèle de prévision de consommation construit dans le volet C Data Scientist.

L’objectif est de dépasser le simple entraînement du modèle pour analyser :
- les performances obtenues ;
- les erreurs de prévision ;
- les périodes où le modèle se trompe le plus ;
- les variables les plus influentes ;
- les limites du modèle ;
- les risques métier ;
- les conditions de mise en production.

Le modèle est utilisé comme un outil d’aide à la décision pour Néovolt, et non comme un système de décision automatique.

In [15]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

In [16]:
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parents[1]

OUTPUT_DIR = ROOT_DIR / "volet-c-data-scientist" / "outputs"
MODELS_DIR = ROOT_DIR / "volet-c-data-scientist" / "models"

DATASET_PATH = OUTPUT_DIR / "dataset_forecast_journalier.csv"
METRICS_PATH = OUTPUT_DIR / "model_forecast_metrics.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "forecast_predictions_test.csv"

OUTPUT_DIR, MODELS_DIR

(WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/volet-c-data-scientist/outputs'),
 WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/volet-c-data-scientist/models'))

In [17]:
# Chargement des données et résultats de modélisation

dataset = pd.read_csv(DATASET_PATH)
metrics = pd.read_csv(METRICS_PATH)
predictions = pd.read_csv(PREDICTIONS_PATH)

dataset["date"] = pd.to_datetime(dataset["date"], errors="coerce")
predictions["date"] = pd.to_datetime(predictions["date"], errors="coerce")

print("Dataset :", dataset.shape)
print("Métriques :", metrics.shape)
print("Prédictions :", predictions.shape)

metrics

Dataset : (717, 24)
Métriques : (3, 4)
Prédictions : (92, 6)


,modele,MAE_kwh,RMSE_kwh,MAPE_%
0,Ridge Regression,3.39,4.17,0.02
1,Random Forest,26.23,48.12,0.17
2,Baseline naïve J-1,1479.04,2371.63,8.84


## 1. Comparaison des performances

Les modèles sont comparés avec trois métriques :

- **MAE** : erreur absolue moyenne en kWh. Elle indique l’erreur moyenne quotidienne.
- **RMSE** : erreur quadratique moyenne. Elle pénalise davantage les grosses erreurs.
- **MAPE** : erreur moyenne en pourcentage. Elle permet une lecture relative de la performance.

Le meilleur modèle est sélectionné selon la MAE, car cette métrique est directement interprétable en kWh pour un décideur métier.

In [18]:
# Classement des modèles par performance

metrics_sorted = metrics.sort_values(by="MAE_kwh", ascending=True)
metrics_sorted

,modele,MAE_kwh,RMSE_kwh,MAPE_%
0,Ridge Regression,3.39,4.17,0.02
1,Random Forest,26.23,48.12,0.17
2,Baseline naïve J-1,1479.04,2371.63,8.84


In [19]:
# Visualisation des métriques

fig = px.bar(
    metrics_sorted,
    x="modele",
    y="MAE_kwh",
    title="Comparaison des modèles selon la MAE",
    labels={
        "modele": "Modèle",
        "MAE_kwh": "MAE (kWh)"
    }
)

fig.show()

In [20]:
# Identification du meilleur modèle

best_model_name = metrics_sorted.iloc[0]["modele"]
best_mae = metrics_sorted.iloc[0]["MAE_kwh"]
best_rmse = metrics_sorted.iloc[0]["RMSE_kwh"]
best_mape = metrics_sorted.iloc[0]["MAPE_%"]

print("Meilleur modèle :", best_model_name)
print("MAE :", best_mae, "kWh")
print("RMSE :", best_rmse, "kWh")
print("MAPE :", best_mape, "%")

Meilleur modèle : Ridge Regression
MAE : 3.39 kWh
RMSE : 4.17 kWh
MAPE : 0.02 %


## 2. Analyse réel vs prédit

La comparaison entre consommation réelle et consommation prédite permet de vérifier si le modèle suit correctement la dynamique temporelle.

Une bonne performance visuelle ne suffit pas : il faut aussi analyser les erreurs jour par jour, notamment les périodes où le modèle sous-estime ou surestime fortement la consommation.

In [21]:
# Courbe réel vs prédit

target = "consommation_totale_kwh"

predictions_long = predictions.melt(
    id_vars=["date"],
    value_vars=[target, "prediction_kwh"],
    var_name="serie",
    value_name="consommation_kwh"
)

fig = px.line(
    predictions_long,
    x="date",
    y="consommation_kwh",
    color="serie",
    markers=True,
    title=f"Consommation réelle vs prédite – {best_model_name}",
    labels={
        "date": "Date",
        "consommation_kwh": "Consommation (kWh)",
        "serie": "Série"
    }
)

fig.show()

In [22]:
# Analyse statistique des erreurs

error_summary = pd.DataFrame([
    {
        "indicateur": "erreur_moyenne_kwh",
        "valeur": round(predictions["erreur_kwh"].mean(), 2)
    },
    {
        "indicateur": "erreur_mediane_kwh",
        "valeur": round(predictions["erreur_kwh"].median(), 2)
    },
    {
        "indicateur": "erreur_absolue_moyenne_kwh",
        "valeur": round(predictions["erreur_absolue_kwh"].mean(), 2)
    },
    {
        "indicateur": "erreur_absolue_max_kwh",
        "valeur": round(predictions["erreur_absolue_kwh"].max(), 2)
    },
    {
        "indicateur": "surestimation_nb_jours",
        "valeur": int((predictions["erreur_kwh"] < 0).sum())
    },
    {
        "indicateur": "sous_estimation_nb_jours",
        "valeur": int((predictions["erreur_kwh"] > 0).sum())
    }
])

error_summary

,indicateur,valeur
0,erreur_moyenne_kwh,-0.37
1,erreur_mediane_kwh,-0.72
2,erreur_absolue_moyenne_kwh,3.39
3,erreur_absolue_max_kwh,10.07
4,surestimation_nb_jours,54.00
5,sous_estimation_nb_jours,38.00


In [23]:
# Graphique des erreurs journalières

fig = px.bar(
    predictions,
    x="date",
    y="erreur_kwh",
    title=f"Erreur de prévision journalière – {best_model_name}",
    labels={
        "date": "Date",
        "erreur_kwh": "Erreur réelle - prédite (kWh)"
    }
)

fig.show()

In [24]:
# Top 10 des plus fortes erreurs absolues

top_errors = (
    predictions
    .sort_values(by="erreur_absolue_kwh", ascending=False)
    .head(10)
)

top_errors

,date,consommation_totale_kwh,prediction_kwh,erreur_kwh,erreur_absolue_kwh,modele
1,2025-10-02,16895.270,16885.198942,10.071058,10.071058,Ridge Regression
7,2025-10-08,17086.870,17078.043632,8.826368,8.826368,Ridge Regression
13,2025-10-14,17538.830,17530.085126,8.744874,8.744874,Ridge Regression
56,2025-11-26,19273.285,19281.788048,-8.503048,8.503048,Ridge Regression
58,2025-11-28,19538.870,19546.904156,-8.034156,8.034156,Ridge Regression
64,2025-12-04,20046.685,20039.446359,7.238641,7.238641,Ridge Regression
87,2025-12-27,16404.435,16411.616471,-7.181471,7.181471,Ridge Regression
77,2025-12-17,20814.680,20821.852914,-7.172914,7.172914,Ridge Regression
59,2025-11-29,15061.470,15068.369379,-6.899379,6.899379,Ridge Regression
67,2025-12-07,16278.200,16271.366818,6.833182,6.833182,Ridge Regression


In [25]:
# Enrichissement des prédictions avec les variables explicatives du dataset de test

test_features = dataset[dataset["split"] == "test"].copy()

predictions_enriched = predictions.merge(
    test_features,
    on=["date", "consommation_totale_kwh"],
    how="left"
)

predictions_enriched.head()

,date,consommation_totale_kwh,prediction_kwh,erreur_kwh,erreur_absolue_kwh,modele,consommation_moyenne_kwh,temp_moyenne_c,temp_min_c,temp_max_c,...,jour_annee,saison_automne,saison_ete,saison_hiver,saison_printemps,conso_lag_1,conso_lag_7,conso_rolling_7,conso_rolling_14,split
0,2025-10-01,16368.420,16368.844831,-0.424831,0.424831,Ridge Regression,23.383457,15.281829,10.755943,20.414371,...,274,True,False,False,False,16403.230,16352.425,15133.002857,14905.052857,test
1,2025-10-02,16895.270,16885.198942,10.071058,10.071058,Ridge Regression,24.136100,14.707514,10.604629,20.267057,...,275,True,False,False,False,16368.420,16010.445,15135.287857,14946.695714,test
2,2025-10-03,17249.075,17243.453674,5.621326,5.621326,Ridge Regression,24.641536,13.122414,8.720071,18.272086,...,276,True,False,False,False,16895.270,16010.975,15261.691429,15009.541786,test
3,2025-10-04,12666.405,12664.165130,2.239870,2.239870,Ridge Regression,18.094864,14.676586,10.288586,20.691957,...,277,True,False,False,False,17249.075,12657.670,15438.562857,15111.072857,test
4,2025-10-05,12675.355,12670.568303,4.786697,4.786697,Ridge Regression,18.107650,13.937729,9.127857,19.508114,...,278,True,False,False,False,12666.405,11965.425,15439.810714,15169.999286,test


In [26]:
# Analyse des erreurs par mois

erreurs_par_mois = (
    predictions_enriched
    .groupby("mois", as_index=False)
    .agg(
        erreur_absolue_moyenne_kwh=("erreur_absolue_kwh", "mean"),
        erreur_moyenne_kwh=("erreur_kwh", "mean"),
        nb_jours=("date", "count")
    )
)

erreurs_par_mois["erreur_absolue_moyenne_kwh"] = erreurs_par_mois["erreur_absolue_moyenne_kwh"].round(2)
erreurs_par_mois["erreur_moyenne_kwh"] = erreurs_par_mois["erreur_moyenne_kwh"].round(2)

erreurs_par_mois

,mois,erreur_absolue_moyenne_kwh,erreur_moyenne_kwh,nb_jours
0,10,3.34,1.84,31
1,11,3.70,-2.30,30
2,12,3.14,-0.71,31


In [27]:
fig = px.bar(
    erreurs_par_mois,
    x="mois",
    y="erreur_absolue_moyenne_kwh",
    title="Erreur absolue moyenne par mois",
    labels={
        "mois": "Mois",
        "erreur_absolue_moyenne_kwh": "Erreur absolue moyenne (kWh)"
    }
)

fig.show()

In [28]:
# Analyse des erreurs par jour de semaine

erreurs_par_jour_semaine = (
    predictions_enriched
    .groupby("jour_semaine", as_index=False)
    .agg(
        erreur_absolue_moyenne_kwh=("erreur_absolue_kwh", "mean"),
        erreur_moyenne_kwh=("erreur_kwh", "mean"),
        nb_jours=("date", "count")
    )
)

erreurs_par_jour_semaine["erreur_absolue_moyenne_kwh"] = erreurs_par_jour_semaine["erreur_absolue_moyenne_kwh"].round(2)
erreurs_par_jour_semaine["erreur_moyenne_kwh"] = erreurs_par_jour_semaine["erreur_moyenne_kwh"].round(2)

erreurs_par_jour_semaine

,jour_semaine,erreur_absolue_moyenne_kwh,erreur_moyenne_kwh,nb_jours
0,0,3.35,-0.89,13
1,1,3.38,-1.32,13
2,2,3.94,-1.66,14
3,3,3.32,0.65,13
4,4,3.60,0.48,13
5,5,2.97,0.02,13
6,6,3.14,0.22,13


## 3. Limites du modèle

Le modèle de prévision est un prototype et présente plusieurs limites :

- la prévision est réalisée au niveau global journalier, pas par compteur ni par zone ;
- le modèle dépend fortement des variables historiques de consommation ;
- les prévisions météo futures ne sont pas intégrées réellement : le prototype utilise les données météo observées du jeu de données ;
- les comportements clients peuvent évoluer dans le temps ;
- les jours exceptionnels, incidents majeurs ou événements externes ne sont pas explicitement modélisés ;
- les performances doivent être validées sur une période plus longue avant industrialisation.

Ces limites ne remettent pas en cause l’intérêt du modèle, mais elles doivent être prises en compte avant toute mise en production.

## 4. Risques métier en cas d’erreur

Dans une infrastructure énergétique, une erreur de prévision peut avoir des conséquences concrètes.

### Sous-estimation de la consommation

Si le modèle prédit une consommation trop faible :
- Néovolt peut sous-préparer ses achats d’énergie ;
- l’entreprise peut devoir acheter de l’énergie en urgence ;
- le réseau peut être plus exposé aux pics ;
- les coûts opérationnels peuvent augmenter.

### Surestimation de la consommation

Si le modèle prédit une consommation trop élevée :
- Néovolt peut surdimensionner ses achats ;
- une partie de l’énergie peut être achetée inutilement ;
- les coûts peuvent également augmenter.

Le modèle doit donc rester un outil d’aide à la décision, avec validation métier et surveillance continue.

## 5. Conditions de mise en production et MLOps

Pour industrialiser ce modèle, Néovolt devrait prévoir :

### Versioning

- versionner le code ;
- versionner les datasets de référence ;
- versionner les paramètres des modèles ;
- tracer les métriques à chaque entraînement.

### Réentraînement

Le modèle devrait être réentraîné régulièrement, par exemple :
- chaque mois ;
- après une évolution importante du parc de compteurs ;
- après un changement de comportement de consommation ;
- après dégradation mesurée des performances.

### Monitoring

En production, il faudrait suivre :
- la MAE réelle au fil du temps ;
- les erreurs par saison ;
- les erreurs lors des jours froids ;
- les dérives de consommation ;
- les dérives météo ;
- les écarts entre prévisions et consommation réelle.

### Déploiement

Dans un premier temps, le modèle pourrait être utilisé en batch quotidien :
- extraction des données récentes ;
- calcul des variables ;
- génération de la prévision J+1 ;
- affichage dans un tableau de bord ;
- validation par un responsable métier.

Le modèle ne doit pas piloter automatiquement une infrastructure critique sans supervision humaine.

## 6. Dimensions transverses

### RGPD

Le modèle travaille à un niveau agrégé journalier.  
Cette approche limite l’exposition des données personnelles par rapport à un modèle par client ou par compteur.

Cependant, les données de consommation restent sensibles. Les accès doivent être limités, tracés et justifiés.

### Biais

Le modèle peut moins bien prévoir certaines périodes ou certains profils si les données historiques ne sont pas représentatives.  
Il faut éviter de considérer la prévision comme une vérité absolue.

### Green IT

Le choix d’un modèle classique comme Ridge ou Random Forest est cohérent avec une démarche sobre :
- entraînement rapide ;
- pas de deep learning inutile ;
- dataset agrégé léger ;
- modèle explicable et maintenable.

### Gouvernance

Les règles de préparation, les variables utilisées, les métriques et les limites doivent être documentées afin de permettre une reprise par une équipe data ou métier.

## Conclusion de l’évaluation

Le modèle de prévision de consommation fournit un prototype crédible pour Néovolt Grid+.

Il permet d’anticiper la consommation totale journalière à partir :
- de l’historique de consommation ;
- des variables calendaires ;
- de la météo ;
- des variables de retard et moyennes mobiles.

Les performances doivent être analysées avec les métriques MAE, RMSE et MAPE, mais aussi avec une lecture métier des erreurs.

Avant toute industrialisation, il serait nécessaire de :
- valider le modèle sur une période plus longue ;
- intégrer de vraies prévisions météo futures ;
- suivre la dérive des performances ;
- mettre en place une supervision métier ;
- documenter les risques liés aux erreurs de prévision.

Le modèle constitue donc une aide à la décision, et non un outil de pilotage automatique.